In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,image_path,label
0,train/img_000001.jpg,1
1,train/img_000002.jpg,1
2,train/img_000003.jpg,1
3,train/img_000004.jpg,0
4,train/img_000005.jpg,1


In [11]:
import torch

epochs = 8
learning_rate = 0.001
batch_size= 32

device = torch.device("cuda")
device

device(type='cuda')

In [12]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import cv2

class MyDataset(Dataset):
    def __init__(self, df, transform, img_size=128):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        label = torch.tensor(row.label, dtype=torch.float32).view(1)
        return img, label

train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.Resize((128, 128)),
    T.ToTensor()
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.ToTensor()
])

from sklearn.model_selection import train_test_split
df_1, df_2 = train_test_split(df_train, random_state=42, test_size=0.2)

train_dataset = MyDataset(df_1, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

test_dataset = MyDataset(df_2, test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [13]:
import torch.nn as nn
import torchvision.models as models

model = models.resnet18(weights='IMAGENET1K_V1')
num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [14]:
def train(model, device, train_loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Training Loss: {total_loss / len(train_loader):.4f}")


def test(model, device, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
    print(f'Test Loss: {test_loss / len(test_loader):.4f}')

In [15]:
for epoch in range(1, epochs+1):
    train(model, device, train_loader, criterion, optimizer, epoch)
    test(model, device, test_loader, criterion)

Epoch 1, Training Loss: 0.3455
Test Loss: 0.1961
Epoch 2, Training Loss: 0.1939
Test Loss: 0.2219
Epoch 3, Training Loss: 0.1328
Test Loss: 0.1964
Epoch 4, Training Loss: 0.1179
Test Loss: 0.1945
Epoch 5, Training Loss: 0.1220
Test Loss: 0.1497
Epoch 6, Training Loss: 0.0841
Test Loss: 0.5969
Epoch 7, Training Loss: 0.1055
Test Loss: 0.3737
Epoch 8, Training Loss: 0.1282
Test Loss: 0.1582


In [16]:
class FinalDataset(Dataset):
    def __init__(self, df, transform, img_size=128):
        self.df = df
        self.img_size = img_size
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(img)
        return img

final_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((128, 128)),
    T.ToTensor()
])

final_dataset = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

In [17]:
model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        results = torch.sigmoid(model(inputs)).cpu()
        preds.append(results)

preds = torch.cat(preds, dim=0).numpy().flatten()
preds = (preds >= 0.5).astype(int)
preds[:10]

array([0, 0, 1, 1, 1, 0, 1, 1, 0, 1])

In [18]:
df_preds = pd.DataFrame({
    "image_path": df_test.image_path,
    "label": preds
})

df_preds

,image_path,label
0,test/img_000615.jpg,0
1,test/img_000616.jpg,0
2,test/img_000617.jpg,1
3,test/img_000618.jpg,1
4,test/img_000619.jpg,1
...,...,...
611,test/img_001226.jpg,1
612,test/img_001227.jpg,1
613,test/img_001228.jpg,0
614,test/img_001229.jpg,0


In [19]:
df_preds.to_csv("submission.csv", index=False)